In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# step 1: Load clean dataset
df_raw = pd.read_csv("../data/telco_churn.csv")

In [3]:
df_raw.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [5]:
df = df_raw.copy()

In [6]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"],errors="coerce")

In [7]:
df.isnull().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [8]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [9]:
# step 2: Drop useless columns
df.drop("customerID", axis=1, inplace=True)

In [10]:
# step 3: encode target variable(churn) 1->yes, 0->no 
df["Churn"] = df["Churn"].map({"Yes":1, "No":0})

In [11]:
df["Churn"].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [12]:
# step 4: Encode categorical features
df = pd.get_dummies(df,drop_first = True)

In [13]:
# STEP 5 — Split Features and Target
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [14]:
# STEP 6 — Train-Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)

In [15]:
# STEP 7 — Scale Numerical Features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [16]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter = 1000)
model.fit(X_train,y_train)

LogisticRegression(max_iter=1000)

In [17]:
#predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]


In [18]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print(confusion_matrix(y_test, y_pred))


[[925 110]
 [162 212]]


In [19]:
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.57      0.61       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



Logistic Regression achieved 
ROC-AUC of 0.84, 
indicating good ability to distinguish churned vs retained customers.

In [20]:
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

ROC-AUC: 0.841778397788628


The Logistic Regression model achieved ROC-AUC of 0.84, indicating strong ability to distinguish churned and retained customers. Further improvement will focus on increasing recall for churn class.

In [21]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.57      0.61       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



In [22]:
model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X_train, y_train)


LogisticRegression(class_weight='balanced', max_iter=1000)

In [23]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

In [24]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.51      0.79      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409



In [25]:
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

ROC-AUC: 0.8414012245214291


In [26]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred_rf))
from sklearn.metrics import roc_auc_score
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.51      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC: 0.8259074117130384


Model Comparison: 
Logistic Regression achieved ROC-AUC of 0.84 with recall of 0.79 for churn class, outperforming Random Forest in identifying churn customers. Since detecting churners is more critical for business retention strategies, Logistic Regression was selected as the final model.

In [27]:
feature_importance = pd.Series(model.coef_[0], index=X.columns)
feature_importance.sort_values(ascending=False).head(10)

InternetService_Fiber optic       0.808897
TotalCharges                      0.494748
StreamingMovies_Yes               0.285948
StreamingTV_Yes                   0.273209
MultipleLines_Yes                 0.201165
PaymentMethod_Electronic check    0.188783
PaperlessBilling_Yes              0.164703
DeviceProtection_Yes              0.058287
SeniorCitizen                     0.056636
PhoneService_Yes                  0.024336
dtype: float64

understanding:

We trained:

model = LogisticRegression(...)
model.fit(X_train, y_train)


After training, the model learned a formula like:

Churn = b0 + b1*x1 + b2*x2 + b3*x3 + ...


Each feature has a weight.

These weights are stored inside:

model.coef_

key Insights:

Customers using fiber optic internet services showed significantly higher churn probability, likely due to higher service expectations and pricing sensitivity. Customers using electronic check payment methods also exhibited higher churn rates, suggesting automatic billing incentives may improve retention

In [28]:
feature_importance.sort_values().head(10)


tenure                                 -1.159048
MonthlyCharges                         -1.039440
Contract_Two year                      -0.617746
Contract_One year                      -0.296706
OnlineSecurity_Yes                     -0.114503
Dependents_Yes                         -0.104292
OnlineBackup_No internet service       -0.099096
OnlineSecurity_No internet service     -0.099096
TechSupport_No internet service        -0.099096
DeviceProtection_No internet service   -0.099096
dtype: float64

Key Insights:

Customers with longer tenure and long-term contracts showed significantly lower churn rates. Customers subscribed to Online Security services also exhibited higher retention, suggesting bundled value-added services improve customer loyalty.

In [29]:
correct = 0

for i in range(50):
    idx = np.random.randint(0, X_test.shape[0])
    sample = X_test[idx].reshape(1, -1)

    pred = model.predict(sample)[0]
    actual = y_test.iloc[idx]

    if pred == actual:
        correct += 1

print("Accuracy on random 50:", correct/50)


Accuracy on random 50: 0.74
